In [1]:
!pip uninstall -y pyautogen autogen-agentchat autogen-ext autogen-core
!pip install pyautogen==0.2.33 pytesseract opencv-python pillow
!apt-get install -y tesseract-ocr

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.4/325.4 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.7/337.7 kB 22.9 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: opencv-python
    Found existing installation: opencv-python 4.13.0.92
    Uninstalling opencv-python-4.13.0.92:
      

^C


In [2]:
# Install if needed
# !pip install pyautogen pytesseract opencv-python pillow
# !apt-get install -y tesseract-ocr

import autogen
import pytesseract
import cv2
from google.colab import files

# -----------------------
# OCR FUNCTION
# -----------------------
def extract_bill_text(image_path):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return pytesseract.image_to_string(gray)


# -----------------------
# Upload Bill Image
# -----------------------
print("Upload bill image")
uploaded = files.upload()

image_path = list(uploaded.keys())[0]

bill_text = extract_bill_text(image_path)

print("\nExtracted Bill Text:\n")
print(bill_text)


# -----------------------
# User Proxy Agent
# -----------------------
user_proxy = autogen.UserProxyAgent(
    name="UserProxy",
    human_input_mode="NEVER",
    code_execution_config={"use_docker": False}
)


# -----------------------
# Bill Processing Agent
# -----------------------
bill_processing_agent = autogen.AssistantAgent(
    name="BillProcessingAgent",
    llm_config=False,
    system_message="Extract items and categorize them."
)


# -----------------------
# Expense Summary Agent
# -----------------------
expense_summary_agent = autogen.AssistantAgent(
    name="ExpenseSummaryAgent",
    llm_config=False,
    system_message="Summarize the categorized expenses."
)


# -----------------------
# Python Function to Process Bill
# -----------------------
def process_bill(text):

    groceries = ["milk", "bread", "apple", "apples"]
    dining = ["burger", "coffee"]
    shopping = ["shampoo"]

    items = []
    summary = {"Groceries":0, "Dining":0, "Shopping":0}

    for line in text.split("\n"):
        if "=" in line and "TOTAL" not in line:
            name, price = line.split("=")
            name = name.strip()
            price = float(price)

            if name.lower() in groceries:
                category="Groceries"
            elif name.lower() in dining:
                category="Dining"
            else:
                category="Shopping"

            items.append((name,category,price))
            summary[category]+=price

    print("\nCategorized Expenses\n")

    for i in items:
        print(i[0],"|",i[1],"|",i[2])

    print("\nExpense Summary\n")

    for k,v in summary.items():
        print(k,":",v)

    print("\nHighest Spending Category:",max(summary,key=summary.get))


# -----------------------
# Run Bill Processing
# -----------------------
process_bill(bill_text)


# -----------------------
# AutoGen Group Chat
# -----------------------
groupchat = autogen.GroupChat(
    agents=[user_proxy, bill_processing_agent, expense_summary_agent],
    messages=[],
    max_round=3
)

manager = autogen.GroupChatManager(
    groupchat=groupchat
)


user_proxy.initiate_chat(
    manager,
    message="Process the bill and summarize expenses."
)

Upload bill image


Saving ChatGPT Image Mar 11, 2026, 09_46_14 AM.png to ChatGPT Image Mar 11, 2026, 09_46_14 AM (1).png

Extracted Bill Text:

SUPERMART

Date: 13-Apr-2024 Time: 11:02 AM
Receipt#: 00012345

 

Milk =60.00
Bread = 40.00
Apples = 80.00
Burger =120.00
Coffee =80.00
Shampoo = 150.00

TOTAL = 530.00


Categorized Expenses

Milk | Groceries | 60.0
Bread | Groceries | 40.0
Apples | Groceries | 80.0
Burger | Dining | 120.0
Coffee | Dining | 80.0
Shampoo | Shopping | 150.0

Expense Summary

Groceries : 180.0
Dining : 200.0
Shopping : 150.0

Highest Spending Category: Dining
UserProxy (to chat_manager):

Process the bill and summarize expenses.

--------------------------------------------------------------------------------

Next speaker: BillProcessingAgent

BillProcessingAgent (to chat_manager):



--------------------------------------------------------------------------------

Next speaker: ExpenseSummaryAgent

ExpenseSummaryAgent (to chat_manager):



--------------------------------------

ChatResult(chat_id=None, chat_history=[{'content': 'Process the bill and summarize expenses.', 'role': 'assistant'}, {'content': '', 'name': 'BillProcessingAgent', 'role': 'user'}, {'content': '', 'name': 'ExpenseSummaryAgent', 'role': 'user'}], summary='', cost={'usage_including_cached_inference': {'total_cost': 0}, 'usage_excluding_cached_inference': {'total_cost': 0}}, human_input=[])